# NER and POS Tagging in Agent Pipelines

## MSA 8700 — Module 8: NLP and Text Processing

---

## Learning Objectives

By the end of this notebook, you will be able to:

1. Perform **Part-of-Speech (POS) tagging** with NLTK and interpret the output
2. Apply POS tags for **rule-based extraction** and **lexical statistics**
3. Perform **Named Entity Recognition (NER)** with spaCy
4. Identify and explain common **entity types** (ORG, GPE, PERSON, DATE, MONEY, etc.)
5. Combine POS tagging and NER in a **structured extraction pipeline**
6. Understand how classical NER/POS models complement **LLMs in agentic workflows**

---

# Part 1: Why Classical NER and POS Tagging?

## LLMs Can Do This — So Why Use Specialized Models?

Large Language Models can perform POS tagging and NER in-context — you can simply ask them to identify entities or tag parts of speech. But classical models like **spaCy** and **NLTK** have important advantages:

| Property | Classical Models (spaCy, NLTK) | LLMs (GPT, Claude) |
|----------|-------------------------------|--------------------|
| **Speed** | Thousands of documents per second | Seconds per document |
| **Cost** | Free at inference time | Per-token API charges |
| **Determinism** | Same input → same output (fixed model) | Stochastic — may vary between calls |
| **Testability** | Easy to evaluate with standard metrics (precision, recall, F1) | Hard to benchmark consistently |
| **Monitoring** | Stable outputs make drift detection straightforward | Output variation makes monitoring harder |

### The Practical Pattern

In production agentic systems, classical models handle the **high-volume, structured extraction** work, and LLMs handle **reasoning and edge cases**:

```
Raw text (emails, news, documents)
        ↓
spaCy NER: extract organizations, people, dates, amounts
        ↓
Structured records stored in database / knowledge graph
        ↓
LLM reasons over the structured data
  ("Which competitors acquired companies in our region last quarter?")
```

## Setup

This notebook uses both **NLTK** (for POS tagging) and **spaCy** (for NER). Let's make sure both are ready.

In [ ]:
# Install if needed:
# pip install nltk spacy
# python -m spacy download en_core_web_sm

import nltk
nltk.download('punkt_tab', quiet=True)
nltk.download('averaged_perceptron_tagger_eng', quiet=True)

import spacy
nlp = spacy.load("en_core_web_sm")

print(f"NLTK version:  {nltk.__version__}")
print(f"spaCy version: {spacy.__version__}")
print(f"spaCy model:   en_core_web_sm")
print("\nSetup complete.")

---

# Part 2: POS Tagging with NLTK

## What Is POS Tagging?

**Part-of-Speech (POS) tagging** assigns a grammatical label to each word in a sentence: noun, verb, adjective, adverb, determiner, etc.

We covered POS tagging briefly in Notebook 02 (Lexical Processing). Here we go deeper into practical applications.

In [ ]:
sentence = "OpenAI acquired a small startup in San Francisco."
tokens = nltk.word_tokenize(sentence)
pos_tags = nltk.pos_tag(tokens)

print("POS Tags:")
for word, tag in pos_tags:
    print(f"  {word:<18} {tag}")

### POS Tag Reference (Penn Treebank)

| Tag | Meaning | Examples |
|-----|---------|----------|
| `NNP` | Proper noun, singular | OpenAI, San, Francisco |
| `VBD` | Verb, past tense | acquired, bought, sold |
| `DT` | Determiner | a, the, an |
| `JJ` | Adjective | small, large, new |
| `NN` | Noun, singular | startup, company, deal |
| `NNS` | Noun, plural | startups, companies |
| `IN` | Preposition | in, on, at, for |
| `RB` | Adverb | quickly, recently, very |
| `VBG` | Verb, gerund | acquiring, running |
| `VBZ` | Verb, 3rd person singular present | acquires, runs |
| `CC` | Coordinating conjunction | and, but, or |
| `.` | Punctuation | . ! ? |

## Application 1: Extracting Adjectives Describing Products

A common business use case: mining product reviews or descriptions for the adjectives customers and marketers use.

In [ ]:
reviews = [
    "The lightweight laptop has an excellent battery and a vivid display.",
    "This affordable headset offers decent sound quality but flimsy construction.",
    "The premium keyboard feels solid and responsive with quiet keys.",
    "A compact router with fast speeds but unreliable firmware updates.",
]

from collections import Counter

adjective_counts = Counter()

for review in reviews:
    tokens = nltk.word_tokenize(review)
    tagged = nltk.pos_tag(tokens)
    
    # Extract all adjectives (JJ, JJR, JJS)
    adjectives = [word.lower() for word, tag in tagged if tag.startswith('JJ')]
    adjective_counts.update(adjectives)

print("Adjectives found across all reviews:\n")
for adj, count in adjective_counts.most_common():
    print(f"  {adj:<16} {count}")

## Application 2: Lexical Statistics from Customer Tickets

What are the most common **verbs** in customer support tickets? This tells you what customers are *doing* (or trying to do).

In [ ]:
tickets = [
    "I tried to log in but the page crashed. I then reset my password.",
    "Cannot access my account. I called support but nobody answered.",
    "I purchased the premium plan but was charged twice. Please refund.",
    "The app froze when I tried to upload a photo. I restarted and tried again.",
    "I want to cancel my subscription. I already emailed twice about this.",
    "Updated the app and now it crashes every time I open it.",
]

verb_counts = Counter()

for ticket in tickets:
    tokens = nltk.word_tokenize(ticket)
    tagged = nltk.pos_tag(tokens)
    
    # Extract verbs (VB, VBD, VBG, VBN, VBP, VBZ)
    verbs = [word.lower() for word, tag in tagged if tag.startswith('VB')]
    verb_counts.update(verbs)

print("Most common verbs in customer tickets:\n")
for verb, count in verb_counts.most_common(15):
    bar = "█" * (count * 3)
    print(f"  {verb:<14} {count:>2}  {bar}")

> **Agentic use case**: A support analytics agent runs POS-based verb extraction over thousands of tickets to build a frequency report. The most common verbs reveal customer pain points ("crash", "cancel", "reset") without any LLM calls.

## Application 3: Extracting Adjective-Noun Pairs for Competitor Analysis

In [ ]:
# Extract [JJ] [NN/NNS] pairs — product descriptors
product_descriptions = [
    "A powerful processor with efficient cooling and a large display.",
    "The sleek design features a durable chassis and bright screen.",
    "An affordable option with reliable performance and long battery.",
    "Premium materials with a lightweight frame and sharp display.",
]

descriptor_counts = Counter()

for desc in product_descriptions:
    tokens = nltk.word_tokenize(desc)
    tagged = nltk.pos_tag(tokens)
    
    for i in range(len(tagged) - 1):
        word1, tag1 = tagged[i]
        word2, tag2 = tagged[i + 1]
        if tag1 == 'JJ' and tag2 in ('NN', 'NNS'):
            descriptor_counts[f"{word1.lower()} {word2.lower()}"] += 1

print("Product descriptors (Adjective + Noun):\n")
for desc, count in descriptor_counts.most_common():
    print(f"  {desc}")

---

# Part 3: Named Entity Recognition (NER) with spaCy

## What Is NER?

**Named Entity Recognition (NER)** identifies and classifies real-world objects mentioned in text: people, organizations, locations, dates, monetary amounts, and more.

Unlike POS tagging (which labels every word), NER focuses on **specific meaningful spans** — the proper nouns, dates, and quantities that you'd want to store in a database.

## Why spaCy for NER?

**spaCy** is a production-grade NLP library optimized for speed and accuracy. Compared to NLTK's NER:

| Feature | NLTK NER | spaCy NER |
|---------|----------|----------|
| **Speed** | Slower (multi-step pipeline) | Fast (optimized C code) |
| **Entity types** | Basic (PERSON, ORG, GPE) | Rich (18+ types) |
| **Multi-word entities** | Requires chunking | Built-in span support |
| **API** | Requires manual chaining | Single `nlp()` call |
| **Models** | Rule-based + MaxEnt | Neural network models |

## Example 1: Basic NER

In [ ]:
doc = nlp("OpenAI acquired a small startup in San Francisco in 2026.")

entities = [(ent.text, ent.label_) for ent in doc.ents]
print("Entities found:")
for text, label in entities:
    print(f"  {text:<20} {label}")

### spaCy Entity Types Reference

| Label | Meaning | Examples |
|-------|---------|----------|
| `PERSON` | People, including fictional | Elon Musk, Marie Curie |
| `ORG` | Organization | OpenAI, Google, United Nations |
| `GPE` | Geo-political entity (country, city, state) | San Francisco, Japan, Texas |
| `LOC` | Non-GPE location | Mount Everest, the Pacific Ocean |
| `DATE` | Absolute or relative date | 2026, March 1st, last quarter |
| `TIME` | Time of day | 3:00 PM, midnight |
| `MONEY` | Monetary value | \$50 million, €100 |
| `PERCENT` | Percentage | 15%, twelve percent |
| `QUANTITY` | Measurement | 100 miles, 3 kilograms |
| `CARDINAL` | Numerals not covered by other types | two, 42, 1000 |
| `ORDINAL` | Ordinal numbers | first, 3rd |
| `PRODUCT` | Product names | iPhone, Windows 11 |
| `EVENT` | Named events | World War II, the Olympics |
| `WORK_OF_ART` | Titles of works | Hamlet, Mona Lisa |
| `LAW` | Named legal documents | the Constitution, GDPR |
| `LANGUAGE` | Languages | English, Mandarin |
| `NORP` | Nationalities, religious/political groups | American, Buddhist, Republican |

## Example 2: Rich Entity Extraction from Business Text

Let's process a realistic business news paragraph and extract all entities.

In [ ]:
news_text = """Microsoft announced on March 1, 2026 that it would invest $2 billion
in Anthropic, the San Francisco-based AI company. CEO Satya Nadella said the
deal would strengthen Microsoft's position in the artificial intelligence market.
The European Commission is expected to review the deal within 90 days. Shares
of Microsoft rose 3.5% on the news."""

doc = nlp(news_text)

print(f"{'Entity':<30} {'Label':<10} {'Description'}")
print("-" * 70)
for ent in doc.ents:
    print(f"{ent.text:<30} {ent.label_:<10} {spacy.explain(ent.label_)}")

Notice how spaCy handles:
- **Multi-word entities**: "San Francisco", "Satya Nadella", "European Commission"
- **Different entity types** from the same text: organizations, people, dates, money, percentages
- **`spacy.explain()`**: Returns a human-readable description of any label

## Example 3: Grouping Entities by Type

For a "deal intelligence agent", you'd want entities organized by category.

In [ ]:
from collections import defaultdict

def extract_entities_by_type(text):
    """Extract entities from text, grouped by type."""
    doc = nlp(text)
    grouped = defaultdict(list)
    for ent in doc.ents:
        # Avoid duplicates within the same category
        if ent.text not in grouped[ent.label_]:
            grouped[ent.label_].append(ent.text)
    return dict(grouped)


result = extract_entities_by_type(news_text)

print("Entities grouped by type:\n")
for label, entities in sorted(result.items()):
    description = spacy.explain(label)
    print(f"  {label} ({description}):")
    for ent in entities:
        print(f"    - {ent}")
    print()

## Example 4: Processing Multiple Documents

A deal intelligence agent doesn't process one article — it processes hundreds. Let's see how to extract entities from a batch of documents and aggregate them.

In [ ]:
news_articles = [
    """Google announced a partnership with Samsung to develop new AI chips.
    The deal, valued at $500 million, was signed in Seoul on February 15, 2026.""",
    
    """Amazon Web Services opened a new data center in Frankfurt, Germany.
    CEO Andy Jassy confirmed the $1.2 billion investment during a press conference.""",
    
    """Apple hired 200 machine learning researchers from DeepMind in London.
    Tim Cook described AI as the company's top priority for 2026.""",
    
    """Meta invested $800 million in its Reality Labs division in Menlo Park.
    Mark Zuckerberg presented the new VR headset at a launch event in Las Vegas.""",
]

# Aggregate entities across all articles
all_orgs = Counter()
all_people = Counter()
all_locations = Counter()
all_money = []

for article in news_articles:
    doc = nlp(article)
    for ent in doc.ents:
        if ent.label_ == 'ORG':
            all_orgs[ent.text] += 1
        elif ent.label_ == 'PERSON':
            all_people[ent.text] += 1
        elif ent.label_ in ('GPE', 'LOC'):
            all_locations[ent.text] += 1
        elif ent.label_ == 'MONEY':
            all_money.append(ent.text)

print("=== Entity Summary Across All Articles ===\n")

print("Organizations mentioned:")
for org, count in all_orgs.most_common():
    print(f"  {org:<30} ({count}x)")

print(f"\nPeople mentioned:")
for person, count in all_people.most_common():
    print(f"  {person:<30} ({count}x)")

print(f"\nLocations mentioned:")
for loc, count in all_locations.most_common():
    print(f"  {loc:<30} ({count}x)")

print(f"\nMonetary values mentioned:")
for money in all_money:
    print(f"  {money}")

> **Agentic use case**: This is exactly what a **deal intelligence agent** does — it processes a stream of news articles, extracts and aggregates entities, and stores them in a knowledge graph or CRM. An LLM then reasons over this structured data: "Which companies invested more than $1 billion? Where are they expanding?"

## Example 5: Entity Context and Surrounding Text

Often you need not just the entity, but the **context** around it — the sentence or clause that mentions it.

In [ ]:
text = """Anthropic raised $3 billion in its latest funding round. The round was
led by Google and included Spark Capital and Salesforce Ventures. CEO Dario
Amodei said the funds would be used to advance AI safety research."""

doc = nlp(text)

print("Entities with their sentence context:\n")
for ent in doc.ents:
    # Find the sentence containing this entity
    sent = ent.sent
    print(f"  Entity:   {ent.text} ({ent.label_})")
    print(f"  Sentence: {sent.text.strip()}")
    print(f"  Position: characters {ent.start_char}–{ent.end_char}")
    print()

The `ent.sent` attribute gives you the full sentence, and `ent.start_char` / `ent.end_char` give you the exact character positions. This is essential for building **entity-in-context** records for a knowledge base.

---

# Part 4: spaCy's Full Pipeline — Beyond NER

When you call `nlp(text)`, spaCy runs a full pipeline that includes tokenization, POS tagging, dependency parsing, lemmatization, and NER — all in one pass. Let's explore the rich token-level annotations.

In [ ]:
doc = nlp("OpenAI acquired a small startup in San Francisco.")

print(f"{'Token':<16} {'POS':<8} {'Tag':<6} {'Dep':<12} {'Lemma':<14} {'Is Stop':<8} {'Entity'}")
print("-" * 80)
for token in doc:
    ent_label = token.ent_type_ if token.ent_type_ else ""
    print(f"{token.text:<16} {token.pos_:<8} {token.tag_:<6} {token.dep_:<12} {token.lemma_:<14} {str(token.is_stop):<8} {ent_label}")

### What Each Attribute Means

| Attribute | Description | Example |
|-----------|-------------|--------|
| `token.text` | The original word | "acquired" |
| `token.pos_` | Coarse POS tag (universal) | "VERB" |
| `token.tag_` | Fine-grained POS tag (Penn Treebank) | "VBD" (past tense verb) |
| `token.dep_` | Dependency relation to head word | "ROOT", "nsubj", "dobj" |
| `token.lemma_` | Dictionary base form | "acquire" |
| `token.is_stop` | Whether it's a stop word | True/False |
| `token.ent_type_` | Entity type (if part of an entity) | "ORG", "GPE" |

## Comparing NLTK POS Tags vs. spaCy POS Tags

In [ ]:
sentence = "OpenAI acquired a small startup in San Francisco."

# NLTK POS tagging
nltk_tokens = nltk.word_tokenize(sentence)
nltk_tags = nltk.pos_tag(nltk_tokens)

# spaCy POS tagging
spacy_doc = nlp(sentence)

print(f"{'Word':<18} {'NLTK Tag':<12} {'spaCy POS':<12} {'spaCy Tag'}")
print("-" * 55)
for (word, nltk_tag), spacy_token in zip(nltk_tags, spacy_doc):
    print(f"{word:<18} {nltk_tag:<12} {spacy_token.pos_:<12} {spacy_token.tag_}")

spaCy provides **two levels** of POS tags:
- `token.pos_` — **Coarse** (Universal Dependencies): `NOUN`, `VERB`, `ADJ`, etc.
- `token.tag_` — **Fine-grained** (Penn Treebank): `NNP`, `VBD`, `JJ`, etc.

The fine-grained tags match what NLTK returns. The coarse tags are simpler and cross-lingual.

---

# Part 5: Building a Structured Extraction Pipeline

Let's combine NER and POS tagging into a complete **information extraction pipeline** — the kind of component a deal intelligence agent would use.

In [ ]:
import re
from collections import defaultdict


def extract_deal_intelligence(text):
    """Extract structured deal information from a news article."""
    doc = nlp(text)
    
    record = {
        'organizations': [],
        'people': [],
        'locations': [],
        'dates': [],
        'monetary_values': [],
        'percentages': [],
        'key_actions': [],     # verbs from the text
        'descriptors': [],     # adjective-noun pairs
    }
    
    # Extract named entities
    for ent in doc.ents:
        if ent.label_ == 'ORG' and ent.text not in record['organizations']:
            record['organizations'].append(ent.text)
        elif ent.label_ == 'PERSON' and ent.text not in record['people']:
            record['people'].append(ent.text)
        elif ent.label_ in ('GPE', 'LOC') and ent.text not in record['locations']:
            record['locations'].append(ent.text)
        elif ent.label_ == 'DATE':
            record['dates'].append(ent.text)
        elif ent.label_ == 'MONEY':
            record['monetary_values'].append(ent.text)
        elif ent.label_ == 'PERCENT':
            record['percentages'].append(ent.text)
    
    # Extract key verbs (non-auxiliary, non-stop)
    for token in doc:
        if token.pos_ == 'VERB' and not token.is_stop and token.dep_ != 'aux':
            if token.lemma_ not in record['key_actions']:
                record['key_actions'].append(token.lemma_)
    
    # Extract adjective-noun pairs
    for i in range(len(doc) - 1):
        if doc[i].pos_ == 'ADJ' and doc[i+1].pos_ == 'NOUN':
            pair = f"{doc[i].text} {doc[i+1].text}"
            if pair not in record['descriptors']:
                record['descriptors'].append(pair)
    
    return record


# Process a news article
article = """Alphabet Inc. announced on February 28, 2026 that Google Cloud
would acquire DataStream Analytics, a fast-growing startup based in Austin,
Texas, for $450 million. CEO Sundar Pichai described the acquisition as a
strategic investment in real-time data processing. The deal is expected to
close in Q2 2026, pending regulatory approval from the Federal Trade
Commission. Google Cloud revenue grew 28% last quarter."""

result = extract_deal_intelligence(article)

print("=== Deal Intelligence Record ===\n")
for field, values in result.items():
    if values:
        print(f"  {field}:")
        for v in values:
            print(f"    - {v}")
        print()

This is a **production-quality extraction record**. Every field was extracted deterministically — no LLM calls, no latency, no API cost. An LLM agent would then reason over these records:

- *"Show me all acquisitions in Texas over $100M"*
- *"Which CEOs have announced AI-related deals this quarter?"*
- *"Compare the deal values across competitors"*

## Processing a Batch of Articles

In [ ]:
articles = [
    """Microsoft invested $2 billion in Anthropic in San Francisco.
    Satya Nadella called it a pivotal moment for AI safety.""",
    
    """Amazon acquired Robotic Systems Inc. in Boston for $890 million.
    The deal will accelerate Amazon's warehouse automation efforts.""",
    
    """Apple opened a new AI research lab in London, hiring 150 researchers
    from local universities. Tim Cook visited the facility on March 1, 2026.""",
]

# Process all articles
all_records = []
for i, art in enumerate(articles, 1):
    record = extract_deal_intelligence(art)
    record['article_id'] = i
    all_records.append(record)

# Aggregate: which organizations appear most?
org_counter = Counter()
for rec in all_records:
    org_counter.update(rec['organizations'])

print("Organizations across all articles:")
for org, count in org_counter.most_common():
    print(f"  {org:<30} mentioned in {count} article(s)")

print(f"\nTotal articles processed: {len(all_records)}")
print(f"Unique organizations found: {len(org_counter)}")
print(f"Total monetary values: {sum(len(r['monetary_values']) for r in all_records)}")

---

# Part 6: Visualizing Entities with displaCy

spaCy includes **displaCy**, a built-in visualizer that highlights entities directly in the text. This is useful for debugging and presentations.

In [ ]:
from spacy import displacy

text = "OpenAI acquired a small startup in San Francisco in 2026 for $50 million."
doc = nlp(text)

# Render entities inline (works in Jupyter notebooks)
displacy.render(doc, style="ent", jupyter=True)

In [ ]:
# Visualize a more complex example
complex_text = """On March 1, 2026, Satya Nadella announced that Microsoft would
invest $2 billion in Anthropic. The San Francisco-based company was founded
by Dario Amodei and Daniela Amodei."""

doc = nlp(complex_text)
displacy.render(doc, style="ent", jupyter=True)

---

# Part 7: NER Determinism and Monitoring

A key advantage of classical NER: **determinism**. Given the same model and the same input, you always get the same output. This makes it possible to build reliable tests and monitoring.

In [ ]:
# Demonstration: spaCy NER is deterministic
text = "Apple announced new products at its headquarters in Cupertino."

# Run NER 5 times — results should be identical
results = []
for i in range(5):
    doc = nlp(text)
    entities = tuple((ent.text, ent.label_) for ent in doc.ents)
    results.append(entities)

# Check that all runs produced identical results
all_identical = all(r == results[0] for r in results)
print(f"Ran NER 5 times on the same input.")
print(f"All results identical: {all_identical}")
print(f"Result: {results[0]}")

In [ ]:
# This determinism enables unit testing:
def test_ner_extracts_org():
    doc = nlp("Google launched a new product.")
    orgs = [ent.text for ent in doc.ents if ent.label_ == 'ORG']
    assert 'Google' in orgs, f"Expected 'Google' in ORGs, got {orgs}"

def test_ner_extracts_money():
    doc = nlp("The deal was worth $5 billion.")
    money = [ent.text for ent in doc.ents if ent.label_ == 'MONEY']
    assert len(money) > 0, f"Expected to find monetary value, got {money}"

def test_ner_extracts_date():
    doc = nlp("The event was held on March 1, 2026.")
    dates = [ent.text for ent in doc.ents if ent.label_ == 'DATE']
    assert len(dates) > 0, f"Expected to find date, got {dates}"

# Run tests
test_ner_extracts_org()
test_ner_extracts_money()
test_ner_extracts_date()
print("All NER tests passed.")

These are **deterministic tests** that will always pass for a given model version. If you upgraded the spaCy model and a test broke, you'd know immediately that entity extraction behavior changed — and could investigate before deploying to production.

This kind of testing is much harder with LLM-based extraction, where outputs can vary between calls.

---

# Part 8: The Big Picture — NER + POS in Agentic Systems

```
┌─────────────────────────────────────────────────────────────────────┐
│                 DEAL INTELLIGENCE AGENT                             │
├─────────────────────────────────────────────────────────────────────┤
│                                                                     │
│  1. DATA INGESTION                                                  │
│     News feeds, emails, SEC filings, press releases                 │
│                          ↓                                          │
│  2. spaCy NER + POS (deterministic, fast, free)                     │
│     • Extract: ORG, PERSON, GPE, DATE, MONEY                        │
│     • Extract: key verbs (acquired, invested, launched)              │
│     • Extract: adjective-noun descriptors                           │
│                          ↓                                          │
│  3. STRUCTURED STORAGE                                              │
│     Knowledge graph / CRM / database                                │
│     {org: "Google", action: "acquire", target: "DataStream",        │
│      amount: "$450M", location: "Austin, TX", date: "2026-02-28"}   │
│                          ↓                                          │
│  4. LLM REASONING (on-demand, per-query)                            │
│     "Which competitors acquired companies in our region              │
│      last quarter with deals over $100M?"                           │
│                                                                     │
└─────────────────────────────────────────────────────────────────────┘
```

spaCy handles the **high-volume, low-cost extraction**. The LLM handles the **high-value, on-demand reasoning**. Together they form a system that is both powerful and cost-effective.

---

# Practice Exercises

Try these exercises to reinforce what you've learned.

## Exercise 1: Verb Frequency Analysis

Use NLTK POS tagging to find the 10 most common verbs across the customer tickets below. What do they reveal about customer pain points?

In [ ]:
customer_tickets = [
    "I tried to reset my password but the link expired before I could click it.",
    "The app crashed when I uploaded a large file. I had to restart.",
    "I purchased the annual plan but I was billed monthly instead.",
    "Cannot connect to the server. I checked my internet and it works fine.",
    "I want to downgrade my plan but the option is missing from settings.",
    "The export feature failed. I need to download my data urgently.",
    "I signed up yesterday but never received the confirmation email.",
    "The search function returns no results even though I know the data exists.",
]

# TODO: Extract all verbs, count frequencies, and print the top 10
# Your code here

## Exercise 2: Entity Extraction from Job Postings

Use spaCy NER to extract organizations, locations, and any monetary values from these job posting snippets.

In [ ]:
job_postings = [
    "Google is hiring a Senior ML Engineer in Mountain View, CA. Salary range: $180,000 - $250,000.",
    "Anthropic seeks a Research Scientist in San Francisco. Competitive compensation starting at $200,000.",
    "Amazon Web Services has an opening for a Data Architect in Seattle, WA.",
    "Meta is looking for a Product Manager in New York City. Base salary: $165,000 - $220,000.",
]

# TODO: 
# 1. Process each job posting with spaCy
# 2. Extract ORG, GPE, and MONEY entities
# 3. Build a structured record for each posting
# 4. Print a summary table

# Your code here

## Exercise 3: Combined Pipeline

Build a function that takes a news article and returns a structured "briefing card" with:
- All organizations and people mentioned
- All monetary values and dates
- The top 3 most important verbs (non-stop, non-auxiliary)
- Any adjective-noun descriptors

In [ ]:
test_article = """Tesla announced record quarterly revenue of $25.2 billion on
January 29, 2026. CEO Elon Musk credited strong demand in China and Europe.
The Austin-based company delivered 495,000 vehicles, a 12% increase over the
previous quarter. Analysts at Goldman Sachs raised their price target to $380."""

# TODO: Write a function that produces a structured briefing card
# Your code here

---

## Summary

| Technique | Library | What It Does | Key Advantage |
|-----------|---------|-------------|---------------|
| **POS Tagging** | NLTK / spaCy | Labels each word with its grammatical role | Enables rule-based extraction patterns |
| **NER** | spaCy | Identifies people, orgs, locations, dates, money, etc. | Structured, deterministic entity extraction |
| **displaCy** | spaCy | Visualizes entities in-line | Debugging and presentation |
| **Batch processing** | spaCy | Process thousands of documents efficiently | Builds knowledge bases at scale |

### Key Takeaways

1. **POS tagging** powers rule-based extraction (adjective-noun pairs, verb frequency) and lexical statistics
2. **spaCy NER** extracts structured entities (ORG, PERSON, GPE, DATE, MONEY) — fast, free, deterministic
3. **Determinism is a feature**: same model + same input = same output, enabling reliable testing and monitoring
4. **Combine NER + POS** for rich extraction pipelines: entities for what's mentioned, POS for how it's described
5. In agentic systems, spaCy handles **high-volume extraction** while LLMs handle **on-demand reasoning** over the structured results